In [ ]:
!pip install -U transformers tokenizers sentencepiece

In [ ]:
!pip install pytorch-crf

In [ ]:
!gdown 1A95kyQzH2YMxizlCeqSgPP2tgKtlayRx

In [ ]:
!gdown 1ATFjncI2J_wSNbCXsBeuJCXKwcbl9-Qo

In [ ]:
!unzip /content/LST20_Corpus.zip

In [ ]:
!unzip /content/super-ai-engineer-ss-6-word-segmentation.zip

In [76]:
import os
import glob
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [ ]:
LST20_TRAIN_DIR = '/content/LST20_Corpus/train'
LST20_EVAL_DIR = '/content/LST20_Corpus/eval'
LST20_TEST_DIR = '/content/LST20_Corpus/test'

TEST_FILE_PATH = '/content/ws_test.txt'
SAMPLE_SUBMISSION_PATH = '/content/ws_sample_submission.csv'
OUTPUT_CSV_NAME = '/content/submission_cnn_bilstm_ultimate.csv'

BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.003
MAX_SEQ_LEN = 300
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [84]:
def get_char_group(char):
    if char.isspace(): return 'SPACE'
    if char.isdigit(): return 'DIGIT'
    if '\u0E01' <= char <= '\u0E2E': return 'CONSONANT'
    if char in 'ะาิีึืุูเแโใไัฤๅฦ': return 'VOWEL'
    if char in '่้๊๋': return 'TONE_MARK'
    if char in '์ๆฯ': return 'THAI_SYMBOL'
    if char.isascii() and char.isalpha(): return 'ENGLISH'
    return 'OTHER'

class LST20Vocab:
    def __init__(self):
        self.char2idx = {'<PAD>': 0, '<UNK>': 1}
        self.group2idx = {'<PAD>': 0}
        self.tag2idx = {'<PAD>': 0, 'O': 1, 'B_WORD': 2, 'I_WORD': 3, 'E_WORD': 4}
        self.idx2tag = {v: k for k, v in self.tag2idx.items()}

    def add_char(self, char):
        if char not in self.char2idx:
            self.char2idx[char] = len(self.char2idx)

    def add_group(self, group):
        if group not in self.group2idx:
            self.group2idx[group] = len(self.group2idx)

def load_lst20(directories, vocab, is_train=True):
    dataset_sentences = []

    for directory in directories:
        if not os.path.exists(directory):
            print(f"Warning: Directory not found: {directory}")
            continue

        file_paths = glob.glob(os.path.join(directory, '*.txt'))
        for file_path in file_paths:
            current_chars, current_groups, current_tags = [], [], []
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip() == '':
                        if current_chars:
                            dataset_sentences.append((current_chars, current_groups, current_tags))
                            current_chars, current_groups, current_tags = [], [], []
                        continue

                    parts = line.strip().split('\t')
                    word = parts[0]
                    if word == '_': word = ' '

                    if word.isspace():
                        for char in list(word):
                            current_chars.append(char)
                            current_groups.append(get_char_group(char))
                            current_tags.append('O')  # ช่องว่างให้เป็น Tag 'O'
                            if is_train:
                                vocab.add_char(char)
                                vocab.add_group(get_char_group(char))
                    else:
                        chars_in_word = list(word)
                        for i, char in enumerate(chars_in_word):
                            current_chars.append(char)
                            current_groups.append(get_char_group(char))

                            if len(chars_in_word) == 1:
                                current_tags.append('B_WORD')
                            elif i == 0:
                                current_tags.append('B_WORD')
                            elif i == len(chars_in_word) - 1:
                                current_tags.append('E_WORD')
                            else:
                                current_tags.append('I_WORD')

                            if is_train:
                                vocab.add_char(char)
                                vocab.add_group(get_char_group(char))
            if current_chars:
                dataset_sentences.append((current_chars, current_groups, current_tags))
    return dataset_sentences

In [ ]:
class WordSegDataset(Dataset):
    def __init__(self, sentences, vocab):
        self.sentences = sentences
        self.vocab = vocab

    def __len__(self): return len(self.sentences)

    def __getitem__(self, idx):
        chars, groups, tags = self.sentences[idx]

        chars = chars[:MAX_SEQ_LEN]
        groups = groups[:MAX_SEQ_LEN]
        tags = tags[:MAX_SEQ_LEN]

        char_ids = [self.vocab.char2idx.get(c, self.vocab.char2idx['<UNK>']) for c in chars]
        group_ids = [self.vocab.group2idx[g] for g in groups]
        tag_ids = [self.vocab.tag2idx[t] for t in tags]

        return torch.tensor(char_ids), torch.tensor(group_ids), torch.tensor(tag_ids)

def pad_collate(batch):
    char_ids = [item[0] for item in batch]
    group_ids = [item[1] for item in batch]
    tag_ids = [item[2] for item in batch]

    char_padded = nn.utils.rnn.pad_sequence(char_ids, batch_first=True, padding_value=0)
    group_padded = nn.utils.rnn.pad_sequence(group_ids, batch_first=True, padding_value=0)
    tag_padded = nn.utils.rnn.pad_sequence(tag_ids, batch_first=True, padding_value=0)

    return char_padded, group_padded, tag_padded

In [ ]:
class CNN_BiLSTM_WordSegmenter(nn.Module):
    def __init__(self, vocab_size, group_vocab_size, num_classes):
        super(CNN_BiLSTM_WordSegmenter, self).__init__()

        # 1. Dual Embeddings
        self.char_emb_dim = 128
        self.group_emb_dim = 32
        self.char_emb = nn.Embedding(vocab_size, self.char_emb_dim, padding_idx=0)
        self.group_emb = nn.Embedding(group_vocab_size, self.group_emb_dim, padding_idx=0)

        total_emb_dim = self.char_emb_dim + self.group_emb_dim
        self.cnn_filters = 128
        self.conv1d = nn.Conv1d(in_channels=total_emb_dim, out_channels=self.cnn_filters, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # 3. BiLSTM Layer (The Mastermind)
        self.bilstm_hidden = 256
        self.bilstm = nn.LSTM(
            input_size=self.cnn_filters,
            hidden_size=self.bilstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(self.bilstm_hidden * 2, num_classes)

    def forward(self, char_ids, group_ids):
        x_char = self.char_emb(char_ids)
        x_group = self.group_emb(group_ids)

        x = torch.cat((x_char, x_group), dim=-1)

        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1d(x))

        x = x.permute(0, 2, 1)

        lstm_out, _ = self.bilstm(x)
        lstm_out = self.dropout(lstm_out)

        logits = self.fc(lstm_out)
        return logits

In [ ]:
if __name__ == "__main__":
    print("📥 1. กำลังสร้าง Vocabulary และโหลดไฟล์ Data ...")
    vocab = LST20Vocab()
    
    all_directories = [LST20_TRAIN_DIR, LST20_EVAL_DIR, LST20_TEST_DIR]
    train_sentences = load_lst20(all_directories, vocab, is_train=True)

    print(f"✅ ได้ข้อมูลทั้งหมด {len(train_sentences):,} ประโยค")
    print(f"✅ จำนวนตัวอักษรในคลัง (Unique Chars): {len(vocab.char2idx)}")

    print("⚙️ 2. กำลังเตรียม PyTorch Data Loader ...")
    train_dataset = WordSegDataset(train_sentences, vocab)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate)

    model = CNN_BiLSTM_WordSegmenter(len(vocab.char2idx), len(vocab.group2idx), len(vocab.tag2idx)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=0) # ไม่คำนวณ Loss ตรง PAD

    print(f"🔥 3. เริ่มฝึกสอนโมเดล (Training on {DEVICE})")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for chars, groups, tags in progress:
            chars, groups, tags = chars.to(DEVICE), groups.to(DEVICE), tags.to(DEVICE)

            optimizer.zero_grad()
            logits = model(chars, groups)

            loss = criterion(logits.view(-1, logits.size(-1)), tags.view(-1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1} | Loss เฉลี่ย: {total_loss/len(train_loader):.4f}")

📥 1. กำลังสร้าง Vocabulary และโหลดไฟล์ Data ...
✅ ได้ข้อมูลทั้งหมด 74,180 ประโยค
✅ จำนวนตัวอักษรในคลัง (Unique Chars): 181
⚙️ 2. กำลังเตรียม PyTorch Data Loader ...
🔥 3. เริ่มฝึกสอนโมเดล (Training on cuda)


Epoch 1/10: 100%|██████████| 580/580 [02:32<00:00,  3.80it/s, loss=0.0435]


Epoch 1 | Loss เฉลี่ย: 0.1329


Epoch 2/10: 100%|██████████| 580/580 [02:35<00:00,  3.74it/s, loss=0.0318]


Epoch 2 | Loss เฉลี่ย: 0.0403


...

Epoch 8 | Loss เฉลี่ย: 0.0219


Epoch 9/10: 100%|██████████| 580/580 [02:34<00:00,  3.75it/s, loss=0.0264]


Epoch 9 | Loss เฉลี่ย: 0.0214


Epoch 10/10: 100%|██████████| 580/580 [02:34<00:00,  3.74it/s, loss=0.0336]

Epoch 10 | Loss เฉลี่ย: 0.0209


In [ ]:
print("\n🔍 4. เริ่มนำโมเดลไปวิเคราะห์ Test Set...")
model.eval()

with open(TEST_FILE_PATH, 'r', encoding='utf-8') as f:
    test_text = f.read().strip()

CHUNK_SIZE = 2000
test_chars = list(test_text)
y_pred = []

print("กำลังวิเคราะห์ทาย Tag ทีละท่อน...")
with torch.no_grad():
    for i in tqdm(range(0, len(test_chars), CHUNK_SIZE)):
        chunk = test_chars[i:i + CHUNK_SIZE]

        char_ids = [vocab.char2idx.get(c, vocab.char2idx['<UNK>']) for c in chunk]
        group_ids = [vocab.group2idx.get(get_char_group(c), vocab.group2idx['<PAD>']) for c in chunk]

        char_tensor = torch.tensor([char_ids]).to(DEVICE)
        group_tensor = torch.tensor([group_ids]).to(DEVICE)

        logits = model(char_tensor, group_tensor)
        preds = torch.argmax(logits, dim=-1)[0].cpu().numpy()

        for p in preds:
            y_pred.append(vocab.idx2tag[p])


🔍 4. เริ่มนำโมเดลไปวิเคราะห์ Test Set...
กำลังวิเคราะห์ทาย Tag ทีละท่อน...


100%|██████████| 19/19 [00:01<00:00, 12.47it/s]


In [ ]:
print("✨ 5. แมปปิ้งผลทายกลับเข้าสู่ฟอร์ม CSV อย่างแม่นยำ...")
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)

char_tag_dict = {}
for i, (char, tag) in enumerate(zip(test_chars, y_pred)):
    
    if tag == 'O' or tag == '<PAD>' or char.isspace(): continue
    char_tag_dict[i + 1] = tag

sample_sub['Predicted'] = sample_sub['Id'].map(char_tag_dict).fillna('B_WORD')
sample_sub.to_csv(OUTPUT_CSV_NAME, index=False)

print(f"🎉 เสร็จสมบูรณ์! ได้ไฟล์คะแนนระดับเทพ: {OUTPUT_CSV_NAME}")

✨ 5. แมปปิ้งผลทายกลับเข้าสู่ฟอร์ม CSV อย่างแม่นยำ...
🎉 เสร็จสมบูรณ์! ได้ไฟล์คะแนนระดับเทพ: /content/submission_cnn_bilstm_ultimate.csv
